# Classification Project: Customer Churn Prediction

**Week 3 Task — Classification Project**

**Problem:** Predict customer churn — a classic, high-stakes imbalanced classification problem (most customers *don't* churn in any given period, which is exactly what makes it tricky to model and to evaluate correctly).

**Note on the data:** this environment has no internet access, so rather than pull a real Kaggle churn CSV, the dataset below is **synthetically generated** to mirror a realistic telecom churn scenario: it uses believable features (tenure, contract type, monthly charges, support calls, tech support, payment method, internet service) and an engineered churn probability driven by realistic risk factors (month-to-month contracts, high support call volume, short tenure, no tech support all increase churn risk). The imbalance (~17% churn rate) matches typical real-world churn rates, so every lesson below about accuracy, precision, recall, and imbalanced data applies exactly as it would to a real dataset — swapping in a real CSV later is a one-line change.

**Goal:** Solve this end-to-end — train/test split, train a model, and evaluate it properly with a confusion matrix plus precision/recall/F1, not just accuracy.

## 1. Generate the Dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              confusion_matrix, classification_report, ConfusionMatrixDisplay)

sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 100
np.random.seed(42)
n = 2000

tenure_months = np.random.gamma(shape=2, scale=15, size=n).clip(0, 72).round(0)
monthly_charges = np.random.normal(65, 25, n).clip(15, 150).round(2)
contract_type = np.random.choice(['Month-to-month', 'One year', 'Two year'], size=n, p=[0.55, 0.25, 0.20])
support_calls = np.random.poisson(1.5, n).clip(0, 10)
has_tech_support = np.random.choice(['Yes', 'No'], size=n, p=[0.4, 0.6])
payment_method = np.random.choice(['Electronic check', 'Mailed check', 'Bank transfer', 'Credit card'], size=n)
internet_service = np.random.choice(['DSL', 'Fiber optic', 'No'], size=n, p=[0.35, 0.45, 0.20])
total_charges = (monthly_charges * tenure_months * np.random.uniform(0.9, 1.0, n)).round(2)

# Realistic risk factors driving churn probability
contract_risk = np.select(
    [contract_type == 'Month-to-month', contract_type == 'One year', contract_type == 'Two year'],
    [1.2, -0.3, -1.0]
)
tech_support_risk = np.where(has_tech_support == 'No', 0.5, -0.4)
tenure_risk = -0.04 * tenure_months
calls_risk = 0.35 * support_calls
charges_risk = 0.01 * (monthly_charges - 65)

logit = -1.9 + contract_risk + tech_support_risk + tenure_risk + calls_risk + charges_risk
prob_churn = 1 / (1 + np.exp(-logit))
churn = (np.random.rand(n) < prob_churn).astype(int)

df = pd.DataFrame({
    'tenure_months': tenure_months, 'monthly_charges': monthly_charges, 'total_charges': total_charges,
    'contract_type': contract_type, 'support_calls': support_calls, 'has_tech_support': has_tech_support,
    'payment_method': payment_method, 'internet_service': internet_service, 'churn': churn
})

df.head()

In [ ]:
print("Class balance:")
print(df['churn'].value_counts())
print(df['churn'].value_counts(normalize=True).round(3))

plt.figure(figsize=(5, 4))
sns.countplot(data=df, x='churn', hue='churn', palette="Set2", legend=False)
plt.title("Class Distribution: Churn vs No Churn")
plt.xticks([0, 1], ['No churn', 'Churn'])
plt.tight_layout()
plt.show()

**This is the crux of the task:** churners are only ~17% of customers. Any evaluation that only looks at overall accuracy can be badly misled by this imbalance — a model can score well just by leaning on the majority class.

## 2. Train/Test Split & Preprocessing

In [ ]:
X = df.drop(columns='churn')
y = df['churn']

numeric_features = ['tenure_months', 'monthly_charges', 'total_charges', 'support_calls']
categorical_features = ['contract_type', 'has_tech_support', 'payment_method', 'internet_service']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(drop='first'), categorical_features)
])

print("Train churn rate:", y_train.mean().round(3))
print("Test churn rate:", y_test.mean().round(3))

`stratify=y` keeps that same ~17% churn rate in both the train and test sets.

## 3. Why Accuracy Lies: The "Always Predict No Churn" Baseline

Before training a real model, check what a **trivial** model would score — predicting the majority class every single time, with no thinking at all.

In [ ]:
dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train, y_train)
y_pred_dummy = dummy.predict(X_test)

print("Baseline ('always predict no churn'):")
print("  Accuracy:", round(accuracy_score(y_test, y_pred_dummy), 3))
print("  Recall on churn class:", round(recall_score(y_test, y_pred_dummy), 3))
print("  Precision on churn class:", round(precision_score(y_test, y_pred_dummy, zero_division=0), 3))

**This is the whole lesson in one cell:** a model that does *nothing* — never even looks at the data — scores **82.8% accuracy**, purely because ~83% of customers don't churn. But its recall on the churn class is **0.0**: it catches not a single churner. If this were used for a real retention campaign, it would flag nobody and every actual churner would be missed. **Accuracy alone would call this the *best* model of the three. It is actually the worst.**

## 4. Model 1 — Logistic Regression

In [ ]:
pipe_lr = Pipeline([
    ('prep', preprocessor),
    ('clf', LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'))
])
pipe_lr.fit(X_train, y_train)
y_pred_lr = pipe_lr.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred_lr):.3f}")
print(classification_report(y_test, y_pred_lr, target_names=['No churn', 'Churn']))

`class_weight='balanced'` tells Logistic Regression to weight the minority (churn) class more heavily during training, since otherwise it would happily lean toward the majority class the same way the baseline did.

## 5. Model 2 — Random Forest

In [ ]:
pipe_rf = Pipeline([
    ('prep', preprocessor),
    ('clf', RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced'))
])
pipe_rf.fit(X_train, y_train)
y_pred_rf = pipe_rf.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred_rf):.3f}")
print(classification_report(y_test, y_pred_rf, target_names=['No churn', 'Churn']))

## 6. Confusion Matrices — Comparing All Three

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred_dummy, ax=axes[0], colorbar=False, cmap="Greys",
                                          display_labels=['No churn', 'Churn'])
axes[0].set_title(f"Baseline (acc={accuracy_score(y_test, y_pred_dummy):.2f}, recall={recall_score(y_test, y_pred_dummy):.2f})")

ConfusionMatrixDisplay.from_predictions(y_test, y_pred_lr, ax=axes[1], colorbar=False, cmap="Blues",
                                          display_labels=['No churn', 'Churn'])
axes[1].set_title(f"Logistic Regression (acc={accuracy_score(y_test, y_pred_lr):.2f}, recall={recall_score(y_test, y_pred_lr):.2f})")

ConfusionMatrixDisplay.from_predictions(y_test, y_pred_rf, ax=axes[2], colorbar=False, cmap="Greens",
                                          display_labels=['No churn', 'Churn'])
axes[2].set_title(f"Random Forest (acc={accuracy_score(y_test, y_pred_rf):.2f}, recall={recall_score(y_test, y_pred_rf):.2f})")

plt.tight_layout()
plt.show()

## 7. Side-by-Side Metrics Table

In [ ]:
results = pd.DataFrame({
    "Model": ["Baseline (majority class)", "Logistic Regression", "Random Forest"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_dummy),
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_rf),
    ],
    "Precision (churn)": [
        precision_score(y_test, y_pred_dummy, zero_division=0),
        precision_score(y_test, y_pred_lr),
        precision_score(y_test, y_pred_rf),
    ],
    "Recall (churn)": [
        recall_score(y_test, y_pred_dummy),
        recall_score(y_test, y_pred_lr),
        recall_score(y_test, y_pred_rf),
    ],
    "F1 (churn)": [
        f1_score(y_test, y_pred_dummy),
        f1_score(y_test, y_pred_lr),
        f1_score(y_test, y_pred_rf),
    ],
})
results.round(3)

## 8. Results Write-Up & Takeaways

| Model | Accuracy | Precision (churn) | Recall (churn) | F1 (churn) |
|---|---|---|---|---|
| Baseline (always "no churn") | **0.828** | 0.00 | **0.00** | 0.00 |
| Logistic Regression | 0.710 | 0.34 | **0.75** | 0.47 |
| Random Forest | 0.848 | 0.72 | **0.19** | 0.30 |

**Key takeaways:**

1. **Accuracy is actively misleading here — dramatically so.** The do-nothing baseline scores the *highest* accuracy of all three (82.8%) — higher than either real model — yet it is completely useless: 0% recall, it catches not one churner. Judging by accuracy alone would rank the useless baseline above both real models.
2. **Random Forest has the highest accuracy of the two real models (84.8%), but by far the worst recall (19%).** It's very conservative: when it does predict churn it's usually right (72% precision), but it misses over 80% of actual churners. For a retention use case, that's a lot of at-risk customers slipping through unflagged.
3. **Logistic Regression trades accuracy for recall, and that trade-off is the right one here.** It flags more customers as at-risk (lower precision, 34%) but catches 75% of true churners — three times Random Forest's recall. For a retention campaign where the cost of a missed churner (lost customer) is much higher than the cost of a false alarm (an unnecessary discount offer), this is the more useful model despite its lower headline accuracy.
4. **`class_weight='balanced'` was essential** for both real models. Without it, they would have drifted toward the baseline's behavior of favoring the majority class, since that minimizes raw error.
5. **Practical recommendation:** for this churn use case, optimize for recall (or F1) on the churn class rather than accuracy, and pick the model/threshold based on the real business cost of missing a churner vs. over-flagging a loyal customer. Here, that points clearly to Logistic Regression over Random Forest.

**Next steps:** tune the classification threshold (rather than the default 0.5) to explicitly trade precision for recall, and try SMOTE or other resampling techniques as an alternative to `class_weight='balanced'`.